# Hierarchical RAG
Here, we load all the RAG chunk JSON files and go through each one to find chunks where the type is "figure". For those, we send the figure’s caption or related content to the GPT-4.1-mini API to get a short summary of what the figure shows, things like the axes, comparisons, or main trends. Then we store that summary back into a new key called image_summary and save everything in the rag_chunks_image_summary folder. This way, all the figures are now represented as text, making it much easier to embed and search later without depending on image embedding models that usually fail to capture the meaning of complex research plots.

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from collections import defaultdict
import re
import hashlib, faiss
import numpy as np

DATA_DIR = Path("data/rag_chunks_image_summary_1000")
all_chunks = []

EMBED_MODEL_ID = "Alibaba-NLP/gte-large-en-v1.5"
BATCH = 64

embed_model = SentenceTransformer(EMBED_MODEL_ID, trust_remote_code=True)

In [ ]:
for file in DATA_DIR.glob("*.rag.chunks.json"):
    paper_id = file.stem.split(".")[0]   # e.g. "2001.08361"
    with open(file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    for ch in chunks:
        all_chunks.append({
            "paper_id": paper_id,
            "id": ch["id"],
            "type": ch["type"],
            "content": ch.get("content", ""),
            "page": ch.get("metadata", {}).get("page"),
            "section": ch.get("metadata", {}).get("section"),
        })

## Level-3 is just like Basic RAG

In [ ]:
def safe_section_id(s):
    if s is None:
        s = "unknown"
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    h = hashlib.md5(s.encode("utf-8")).hexdigest()[:12]
    return h

level3 = [
    {
        "uid": f"{x['paper_id']}__{x['id']}",
        "paper_id": x["paper_id"],
        "section": x["section"],
        "section_id": safe_section_id(x["section"]),
        "text": x["content"]
    }
    for x in all_chunks
    if x["content"].strip()
]

with open("data/metadata/meta_l3_1000.json", "w") as f:
    json.dump(level3, f, indent=2)

In [ ]:
len(all_chunks), len(level3)

In [ ]:
def embed_items(items):
    texts = [x["text"] for x in items]
    embs = embed_model.encode(texts, batch_size=BATCH, show_progress_bar=True, normalize_embeddings=True)
    return np.array(embs).astype("float32")

In [ ]:
# secton level

from collections import defaultdict

section_map = defaultdict(list)

for x in all_chunks:
    page = x["page"]
    if page is None:
        continue
    key = (x["paper_id"], f"page_{page}")
    section_map[key].append(x["content"])


level2 = []

for (pid, page), texts in section_map.items():
    sec_id = page              # e.g. "page_1"
    merged = "\n".join(texts)[:4000]

    level2.append({
        "uid": f"{pid}__{sec_id}",
        "paper_id": pid,
        "section": sec_id,
        "section_id": sec_id,
        "text": merged
    })

with open("data/metadata/meta_l2_1000.json", "w") as f:
    json.dump(level2, f, indent=2)

In [ ]:
# document level

paper_map = defaultdict(list)

for x in all_chunks:
    if x["section"] in ("Abstract", "Summary", "Introduction"):
        paper_map[x["paper_id"]].append(x["content"])

level1 = []

for pid, texts in paper_map.items():
    merged = "\n".join(texts)[:4000]
    level1.append({
        "uid": pid,
        "paper_id": pid,
        "text": merged
    })

In [ ]:
len(all_chunks), len(level3), len(level2), len(level1)

# Embed all levels

In [ ]:
def embed_items(items):
    texts = [x["text"] for x in items]
    embs = embed_model.encode(texts, batch_size=BATCH, show_progress_bar=True, normalize_embeddings=True)
    return np.array(embs).astype("float32")

In [ ]:
emb_l1 = embed_items(level1)
emb_l2 = embed_items(level2)
emb_l3 = embed_items(level3)

In [ ]:
# save embeddings

np.save("data/embeddings/emb_l1_1000.npy", emb_l1)
np.save("data/embeddings/emb_l2_1000.npy", emb_l2)
np.save("data/embeddings/emb_l3_1000.npy", emb_l3)

# Build 3 FAISS indices

In [ ]:
FAISS_DIR = Path("data/faiss")
FAISS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
emb_l1 = np.load("data/embeddings/emb_l1_1000.npy")
emb_l2 = np.load("data/embeddings/emb_l2_1000.npy")
emb_l3 = np.load("data/embeddings/emb_l3_1000.npy")

idx_l1 = faiss.read_index("data/faiss/index_l1_1000.faiss")
idx_l2 = faiss.read_index("data/faiss/index_l2_1000.faiss")
idx_l3 = faiss.read_index("data/faiss/index_l3_1000.faiss")

level1 = json.load(open("data/metadata/meta_l1_1000.json"))
level2 = json.load(open("data/metadata/meta_l2_1000.json"))
level3 = json.load(open("data/metadata/meta_l3_1000.json"))

d = emb_l1.shape[1]

def build_hnsw(embs):
    idx = faiss.IndexHNSWFlat(d, 32)
    idx.hnsw.efConstruction = 200
    idx.add(embs)
    return idx

# idx_l1 = build_hnsw_index(emb_l1)
# idx_l2 = build_hnsw_index(emb_l2)
# idx_l3 = build_hnsw_index(emb_l3)

In [ ]:
# --------------------------------------------------
# 2. LEVEL-1: global paper index
# --------------------------------------------------

idx_l1 = build_hnsw(emb_l1)
faiss.write_index(idx_l1, str(FAISS_DIR / "l1_global.faiss"))

with open("data/metadata/meta_l1_1000.json", "w") as f:
    json.dump(level1, f, indent=2)

In [ ]:
# --------------------------------------------------
# 3. LEVEL-2: per-paper page indices
# --------------------------------------------------

l2_by_paper = defaultdict(list)

for i, x in enumerate(level2):
    l2_by_paper[x["paper_id"]].append(i)

l2_manifest = {}

for paper_id, idxs in l2_by_paper.items():
    sub_embs = emb_l2[idxs]
    sub_index = build_hnsw(sub_embs)

    out_path = FAISS_DIR / f"l2_{paper_id}.faiss"
    faiss.write_index(sub_index, str(out_path))

    # store mapping so query-time lookup is trivial
    l2_manifest[paper_id] = {
        "index_path": str(out_path),
        "meta_indices": idxs
    }

with open("data/metadata/l2_manifest.json", "w") as f:
    json.dump(l2_manifest, f, indent=2)



In [ ]:
# --------------------------------------------------
# 4. LEVEL-3: per-(paper,page) chunk indices
# --------------------------------------------------
import re
import hashlib



l3_by_key = defaultdict(list)

for i, x in enumerate(level3):
    safe_section = safe_section_id(x["section"])
    key = f"{x['paper_id']}__{safe_section}"   # ← USE SAFE NAME
    l3_by_key[key].append(i)


l3_manifest = {}



for key, idxs in l3_by_key.items():
    sub_embs = emb_l3[idxs]
    sub_index = build_hnsw(sub_embs)

    out_path = FAISS_DIR / f"l3_{key}.faiss"
    faiss.write_index(sub_index, str(out_path))

    paper_id, sec_id = key.split("__", 1)

    l3_manifest[key] = {
    "index_path": str(out_path),
    "meta_indices": idxs,
    "paper_id": x["paper_id"],
    "section_id": sec_id
    }

with open("data/metadata/l3_manifest.json", "w") as f:
    json.dump(l3_manifest, f, indent=2)



In [ ]:
print("len(level3):", len(level3))
print("emb_l3 shape:", emb_l3.shape)
print("max index used:", max(i for idxs in l3_by_key.values() for i in idxs))

# Adaptive Depth-Controller

In [ ]:
def embed_query(q):
    return embed_model.encode([q], normalize_embeddings=True).astype("float32")


def choose_depth(query):
    q = query.lower()
    if re.search(r"(figure|eqn|equation|derive|value|algorithm)", q):
        return 3
    if len(query.split()) <= 6:
        return 1
    if len(query.split()) <= 15:
        return 2
    return 3

In [ ]:
def search_spi(query, k1=5, k2=10, k3=20):
    depth = choose_depth(query)
    qemb = embed_query(query)

    # ---- LEVEL 1 ----
    D1, I1 = idx_l1.search(qemb, k1)
    papers = [level1[i] for i in I1[0]]

    if depth == 1:
        return {"level": 1, "papers": papers}

    # ---- LEVEL 2 ----
    cand_pids = {p["paper_id"] for p in papers}
    l2_filtered = [i for i, x in enumerate(level2) if x["paper_id"] in cand_pids]
    if not l2_filtered:
        return {"level": 1, "papers": papers}

    # TEMP FAISS sub-index for level 2
    dim2 = emb_l2.shape[1]
    sub2 = faiss.IndexFlatIP(dim2)
    sub2.add(emb_l2[l2_filtered])

    D2, I2 = sub2.search(qemb, k2)
    sections = [level2[l2_filtered[i]] for i in I2[0]]

    if depth == 2:
        return {"level": 2, "papers": papers, "sections": sections}

    # ---- LEVEL 3 ----
    cand_sections = {(s["paper_id"], s["section"]) for s in sections}
    l3_filtered = [i for i, x in enumerate(level3)
                   if (x["paper_id"], x["section"]) in cand_sections]

    if not l3_filtered:
        return {"level": 2, "papers": papers, "sections": sections}

    # TEMP FAISS sub-index for level 3
    dim3 = emb_l3.shape[1]
    sub3 = faiss.IndexFlatIP(dim3)
    sub3.add(emb_l3[l3_filtered])

    D3, I3 = sub3.search(qemb, k3)
    chunks = [level3[l3_filtered[i]] for i in I3[0]]

    return {"level": 3, "papers": papers, "sections": sections, "chunks": chunks}


In [ ]:
query = "What is the scaling law relationship between compute and test loss?"
result = search_spi(query)

print("=== Retrieval Level Chosen ===")
print(result["level"])

In [ ]:
print("\n=== PAPERS (Level 1) ===")
for p in result.get("papers", []):
    print(f"[{p['paper_id']}]")
    print(p['text'], "...\n")

In [ ]:
print("\n=== SECTIONS (Level 2) ===")
for s in result.get("sections", []):
    print(f"[{s['paper_id']} - {s['section']}]")
    print(s['text'], "...\n")

In [ ]:
print("\n=== CHUNKS (Level 3) ===")
for c in result.get("chunks", []):
    print(f"[{c['uid']}]")
    print(c['text'][:250], "...\n")

# Generation

In [ ]:
SYSTEM = (
  "Answer ONLY from <chunk> context. Read EVERY chunk.\n"
  "Step 1: Extract EVERY distinct method/model by its PROPER NAME as written "
  "(use exact capitalization), then give a 1-line description.\n"
  "Step 2: Write a concise answer that covers each named item.\n"
  "If nothing relevant: Not found in the given context.\n"
  "Paraphrase descriptions; DO NOT rename methods. Cite as [DOC:doc_id, p:page]."
)

GEN_CFG = dict(
    max_new_tokens=512,        # maximum tokens model can generate in the output
    temperature=0.5,           # randomness; lower = more deterministic, higher = more creative
    top_p=0.9,                 # nucleus sampling; model samples only from top 90% probability mass
    do_sample=False,            # if True, enables stochastic sampling (else it picks argmax each time)
    repetition_penalty=1.01,   # penalizes repeating same tokens; >1 discourages loops/redundancy
    no_repeat_ngram_size=8,    # forbids repeating any 8-token sequence exactly
)

In [ ]:
def build_context(result, max_chunks=10):
    chunks = result.get("chunks", [])[:max_chunks]
    sections = result.get("sections", [])[:3]  # optional

    context_blocks = []

    # Add sections (higher-level summaries)
    for sec in sections:
        context_blocks.append(f"<chunk id='{sec['uid']}'>\n{sec['text']}\n</chunk>")

    # Add fine chunks
    for ch in chunks:
        context_blocks.append(f"<chunk id='{ch['uid']}'>\n{ch['text']}\n</chunk>")

    return "\n\n".join(context_blocks)

In [ ]:
def build_prompt(query, context):
    return f"{SYSTEM}\n\n<context>\n{context}\n</context>\n\nUser question: {query}\nAnswer:"


In [ ]:
import torch, faiss, ujson as json
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEN_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

generator_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)

In [ ]:
def generate_answer(prompt):
    inputs = tok(prompt, return_tensors="pt").to(generator_model.device)
    output = generator_model.generate(**inputs, **GEN_CFG)
    return tok.decode(output[0], skip_special_tokens=True)

In [ ]:
def answer_question(query, k1=5, k2=10, k3=20):
    # retrieval
    result = search_spi(query, k1=k1, k2=k2, k3=k3)

    # build context
    context = build_context(result)

    # build prompt
    prompt = build_prompt(query, context)

    # generate
    answer = generate_answer(prompt)

    return {
        "retrieval_level": result["level"],
        "context": context,
        "prompt": prompt,
        "answer": answer
    }

In [ ]:
resp = answer_question("What methods exist to accelerate token generation during inference without retraining the language model")
print(resp["answer"])